In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import json
from xgboost import XGBClassifier

# Load datasets
train = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/train.csv')
test = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/test.csv')
meta = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/metaData.csv')
# Logs
wa = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/whatsapp_activity.csv')
bot = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/teleco_call_back.csv')
human = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/call_placed.csv')
field = pd.read_csv('/kaggle/input/cred-resolve-intelligence-challenge-next-best/mobile_app_data.csv')

In [ ]:
# WhatsApp Features
wa_features = wa.groupby('lead_code').agg(
    wa_count=('status', 'count'),
    wa_read_rate=('read_at', lambda x: x.notnull().mean()),
    wa_last_fail=('status', lambda x: 1 if x.iloc[-1] == 'FAILED' else 0)
).reset_index()

# Bot Features: Extracting Intent from JSON
def extract_intent(json_str):
    try:
        return json.loads(json_str).get('intent', 'unknown')
    except:
        return 'none'

bot['intent'] = bot['transcript_json'].apply(extract_intent)
bot_features = bot.groupby('lead_code').agg(
    bot_avg_duration=('duration', 'mean'),
    bot_ptp_intent=('intent', lambda x: (x == 'PTP').sum()) # Promise to Pay
).reset_index()

In [ ]:
# Field Visit Features
field_features = field.groupby('lead_code').agg(
    total_visits=('visit_date', 'count'),
    last_visit_result=('result', 'last')
).reset_index()

In [ ]:
def build_master_df(df):
    # 1. Join with Meta
    df = df.merge(meta, on='lead_code', how='left')
    
    # 2. Join with logs
    df = df.merge(wa_features, on='lead_code', how='left')
    df = df.merge(bot_features, on='lead_code', how='left')
    df = df.merge(field_features, on='lead_code', how='left')
    
    # --- CRITICAL FIX START ---
    
    # Identify categorical/string columns BEFORE fillna
    # because fillna(0) turns them into "object" (mixed int/str)
    cat_cols = ['suggested_action', 'dpd_bucket', 'state', 'last_visit_result']
    
    for col in cat_cols:
        if col in df.columns:
            # Fill missing categorical values with a string 'UNKNOWN' instead of 0
            df[col] = df[col].fillna('UNKNOWN').astype(str)
            
    # Now fill numeric columns with 0
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    
    # --- CRITICAL FIX END ---

    # One-Hot Encoding
    df = pd.get_dummies(df, columns=['suggested_action', 'dpd_bucket', 'state'])
    
    return df

# Re-run the build
train_final = build_master_df(train)
test_final = build_master_df(test)

In [ ]:
# Separate features and target
y = train_final['TARGET']
X = train_final.drop(['id', 'lead_code', 'TARGET'], axis=1)
X_test = test_final.drop(['id', 'lead_code'], axis=1)

# Ensure last_visit_result (if still there) is handled
if 'last_visit_result' in X.columns:
    X['last_visit_result'] = X['last_visit_result'].astype('category')
    X_test['last_visit_result'] = X_test['last_visit_result'].astype('category')

# Align: Add missing columns to test as 0, remove extra columns in test
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"Final training features: {X.shape[1]}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# 1. Create bins for Stratified Splitting
y_bins = pd.cut(y, bins=10, labels=False)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

final_preds = np.zeros(len(X_test))

print("Starting Cross-Validation Training...")

for i, (train_idx, val_idx) in enumerate(skf.split(X, y_bins)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # FIX: Define early_stopping_rounds here, inside the constructor
    m = XGBRegressor(
        n_estimators=2000, 
        learning_rate=0.02, 
        max_depth=5,
        objective='reg:logistic',
        tree_method='hist',
        enable_categorical=True,
        n_jobs=-1,
        early_stopping_rounds=50, # Moved here
        eval_metric='mae'          # Added metric to monitor
    )
    
    # 2. Fit the model (removed early_stopping_rounds from here)
    m.fit(
        X_train, y_train, 
        eval_set=[(X_val, y_val)], 
        verbose=False
    )
    
    final_preds += m.predict(X_test) / 5
    print(f"Fold {i+1} complete.")

print("All folds finished!")

In [ ]:
from xgboost import XGBRegressor

# Initialize the model
# Using 'hist' for speed and 'reg:logistic' for [0,1] probability range
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    objective='reg:logistic',
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

# Fit on the entire training dataset
print("Training model...")
model.fit(X, y)
print("Training complete!")

In [ ]:
# Predict probabilities
# Because we used reg:logistic, the output is already a probability between 0 and 1
test_predictions = model.predict(X_test)

# Create the final dataframe
submission = pd.DataFrame({
    'id': test['id'],
    'TARGET': test_predictions
})

# Guardrail: Ensure no values are outside [0, 1] due to floating point errors
submission['TARGET'] = submission['TARGET'].clip(0, 1)

# Save to CSV
submission.to_csv('submission1.csv', index=False)
print("Submission file 'submission1.csv' created successfully!")